In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
import statsmodels.formula.api as smf

# Chargement
traitement = pd.read_csv("/home/onyxia/stage_2A/data/traitement_clean.csv")
controls = pd.read_csv("/home/onyxia/stage_2A/api_geo_checkpoint.csv")
merge4 = pd.read_csv("/home/onyxia/stage_2A/merge4_filtre.csv")

# Nettoyage et jointures
controls_clean = pd.merge(merge4, controls, left_on='bvdid', right_on='siren_clean', how='inner')
traitement['year'] = pd.to_numeric(traitement['year'])
controls_clean['year'] = pd.to_numeric(controls_clean['year'])

In [2]:


# 1. Forcer l'affichage de TOUTES les lignes (pas de coupure au milieu)
pd.set_option('display.max_rows', None)

# 2. Forcer l'affichage de TOUTES les colonnes
pd.set_option('display.max_columns', None)

# 3. Forcer l'affichage de tout le texte sans le tronquer (ex: les longs noms de pôles)
pd.set_option('display.max_colwidth', None)

In [3]:
traitement["temps_relatif"].value_counts()

temps_relatif
 1     3246
 0     3163
 2     3112
-1     2892
 3     2889
-2     2658
-3     2479
 4     2456
-4     2330
-5     2154
-6     2052
 5     1939
-7     1915
-8     1798
-9     1662
-10    1503
-11    1409
 6     1322
-12    1308
-13    1179
-14     949
 7      780
-15     682
-16     447
 8      302
-17     270
-18     126
 9       77
-19      40
-20       8
 10       6
-22       4
-21       4
-23       2
-24       2
-25       2
-26       2
Name: count, dtype: int64

In [4]:
traitement=traitement[traitement["year"]>2000]

In [5]:
def critere_coherence(x):
    temps = x["temps_relatif"]
    # Compte le nombre d'observations avant 0
    nombre_avant = (temps < 0).sum()
    # Compte le nombre d'observations à 0 ou plus
    nombre_apres = (temps >= 0).sum()
    
    # On garde si les deux conditions sont remplies
    return (nombre_avant >= 2) and (nombre_apres >= 2)

# Application du filtre
traitement_filtre = traitement.groupby("bvdid").filter(critere_coherence)

In [6]:
traitement_filtre["bvdid"].nunique()

1779

préparation des controles

In [7]:
# 1. On identifie les bvdid à exclure (ceux présents dans 'traitement')
bvdid_a_exclure = traitement["bvdid"].unique()

# 2. On filtre 'controls_clean'
# Le tilde ~ signifie "NON", donc on garde les bvdid qui NE SONT PAS dans la liste
controls_final = controls_clean[~controls_clean["bvdid"].isin(bvdid_a_exclure)]

In [8]:
# On garde uniquement les lignes où la devise n'est pas "USD"
controls_final = controls_final[controls_final["original_currency"] != "USD"]

In [9]:
# Pour supprimer plusieurs colonnes à la fois
controls_final = controls_final.drop(columns=['country_code', 'filing_type', 'number_of_months', 'original_currency', 'exchange_rate_from_original_currency', 'commune_insee'])

In [10]:
# Pour supprimer plusieurs colonnes à la fois
traitement_filtre = traitement_filtre.drop(columns=['country_code', 'filing_type', 'number_of_months', 'original_currency', 'exchange_rate_from_original_currency'])

traitement des doublons

In [11]:
# 1. Lister les colonnes qu'on veut sommer
colonnes_somme = ['montant_feder', 'cout_total']

# 2. Identifier les autres colonnes numériques
colonnes_num = [col for col in traitement_filtre.select_dtypes(include='number').columns 
                if col not in ['bvdid', 'temps_relatif'] + colonnes_somme]

# 3. Créer le dictionnaire de règles
agg_rules = {col: 'sum' for col in colonnes_somme}
agg_rules.update({col: 'first' for col in colonnes_num})

# 4. ASTUCE : On ajoute une colonne de comptage de projets
# On prend n'importe quelle colonne (ex: 'libelle_projet') pour compter les occurrences
if 'libelle_projet' in traitement_filtre.columns:
    agg_rules['libelle_projet'] = 'count'

# 5. Appliquer
traitement_propre = traitement_filtre.groupby(['bvdid', 'temps_relatif']).agg(agg_rules).reset_index()

# 6. Renommer pour la clarté
traitement_propre = traitement_propre.rename(columns={'libelle_projet': 'nb_projets_annee'})

In [14]:
traitement_propre[traitement_propre["bvdid"]==15871023]

,bvdid,temps_relatif,montant_feder,cout_total,code_departement,annee,siren,closing_date,total_assets,number_of_employees,operating_revenue_turnover,sales,status_date,date_of_incorporation,delisted_date,ipo_date,year,post,nace_rev_2_core_code_4_digits,nb_projets_annee
16,15871023,-8,82964.77,518362.19,76,2010,15871023,20021231,1684000.0,20.0,3231000.0,3363000.0,NaN,196601.0,NaN,NaN,2002,0,2899,2
17,15871023,-7,82964.77,518362.19,76,2010,15871023,20031231,2467648.0,19.0,4056377.0,3716919.0,NaN,196601.0,NaN,NaN,2003,0,2899,2
18,15871023,-6,82964.77,518362.19,76,2010,15871023,20041231,2399876.0,19.0,4593680.0,4697327.0,NaN,196601.0,NaN,NaN,2004,0,2899,2
19,15871023,-5,82964.77,518362.19,76,2010,15871023,20051231,2846080.0,21.0,4837910.0,4731253.0,NaN,196601.0,NaN,NaN,2005,0,2899,2
20,15871023,-3,82964.77,518362.19,76,2010,15871023,20071231,2974000.0,35.0,4652000.0,4361000.0,NaN,196601.0,NaN,NaN,2007,0,2899,2
21,15871023,0,82964.77,518362.19,76,2010,15871023,20101231,3988076.0,37.0,6130121.0,5606873.0,NaN,196601.0,NaN,NaN,2010,1,2899,2
22,15871023,1,82964.77,518362.19,76,2010,15871023,20111231,4017000.0,NaN,6500000.0,6500000.0,NaN,196601.0,NaN,NaN,2011,1,2899,2
23,15871023,2,82964.77,518362.19,76,2010,15871023,20121231,3596332.0,NaN,6109556.0,6138064.0,NaN,196601.0,NaN,NaN,2012,1,2899,2
24,15871023,3,82964.77,518362.19,76,2010,15871023,20131231,3728109.0,NaN,5439771.0,5059093.0,NaN,196601.0,NaN,NaN,2013,1,2899,2
25,15871023,4,82964.77,518362.19,76,2010,15871023,20141231,3846449.0,NaN,5968437.0,5656304.0,NaN,196601.0,NaN,NaN,2014,1,2899,2


In [25]:
import pandas as pd
import numpy as np

def calculer_indicateurs_taux(x):
    # On isole la période pré-traitement
    pre = x[(x["temps_relatif"] < 0) & (x["temps_relatif"] >= -4)].copy().sort_values("temps_relatif")
    
    if len(pre) >= 2:
        # Calcul des taux
        pre['croissance_assets'] = pre['total_assets'].pct_change()
        pre['croissance_sales'] = pre['sales'].pct_change()
        
        # On renvoie un dictionnaire au lieu d'une Série
        return {
            'valeur_assets_T_minus_1': pre['total_assets'].iloc[-1],
            'valeur_sales_T_minus_1': pre['sales'].iloc[-1],
            'moyenne_croissance_assets': pre['croissance_assets'].mean(),
            'moyenne_croissance_sales': pre['croissance_sales'].mean()
        }
    else:
        # On renvoie des None si pas assez de données
        return {
            'valeur_assets_T_minus_1': np.nan,
            'valeur_sales_T_minus_1': np.nan,
            'moyenne_croissance_assets': np.nan,
            'moyenne_croissance_sales': np.nan
        }

# Application : on utilise .apply() et on transforme le résultat en DataFrame proprement
# On transforme la liste de dictionnaires en DataFrame
results = traitement_propre.groupby("bvdid").apply(calculer_indicateurs_taux)

# Si results est une Series de dictionnaires, on le convertit en DataFrame :
baseline_data = pd.DataFrame(results.tolist(), index=results.index).reset_index()

# Résultat : un beau DataFrame avec bvdid en colonne et vos indicateurs
print(baseline_data.head())


/tmp/ipykernel_103199/2862231958.py:10: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  pre['croissance_assets'] = pre['total_assets'].pct_change()
/tmp/ipykernel_103199/2862231958.py:10: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  pre['croissance_assets'] = pre['total_assets'].pct_change()
/tmp/ipykernel_103199/2862231958.py:10: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  pre['croissance_assets'] 

      bvdid  valeur_assets_T_minus_1  valeur_sales_T_minus_1  \
0   7180516               24449660.0              10223813.0   
1  15871023                      NaN                     NaN   
2  16150419                      NaN              15000000.0   
3  16750770                 967243.0               1704785.0   
4  35650274                      NaN               1570000.0   

   moyenne_croissance_assets  moyenne_croissance_sales  
0                  -0.004031                  1.411618  
1                        NaN                       NaN  
2                  -0.004297                  0.122082  
3                  -0.005685                 -0.013161  
4                  -0.004368                  0.059378  


/tmp/ipykernel_103199/2862231958.py:11: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  pre['croissance_sales'] = pre['sales'].pct_change()
/tmp/ipykernel_103199/2862231958.py:11: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  pre['croissance_sales'] = pre['sales'].pct_change()
/tmp/ipykernel_103199/2862231958.py:31: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping

In [26]:
baseline_data.head()

,bvdid,valeur_assets_T_minus_1,valeur_sales_T_minus_1,moyenne_croissance_assets,moyenne_croissance_sales
0,7180516,24449660.0,10223813.0,-0.004031,1.411618
1,15871023,NaN,NaN,NaN,NaN
2,16150419,NaN,15000000.0,-0.004297,0.122082
3,16750770,967243.0,1704785.0,-0.005685,-0.013161
4,35650274,NaN,1570000.0,-0.004368,0.059378


In [24]:
baseline_data[baseline_data["bvdid"]==15871023]

,bvdid,level_1,0
4,15871023,valeur_assets_T_minus_1,NaN
5,15871023,valeur_sales_T_minus_1,NaN
6,15871023,moyenne_croissance_assets,NaN
7,15871023,moyenne_croissance_sales,NaN
8,15871023,moyenne_nb_projets,NaN
